# SNAPTOCK — fine-tune PP-OCRv5 on handwritten nota

Runs end to end on a GPU box: download → prepare → verify → train → export → evaluate.

**Before running**, from a shell in the repo root:

```bash
cp .env.example .env          # set ROBOFLOW_API_KEY
pip install -r requirements.txt
pip install paddlepaddle-gpu==3.0.0   # MUST match your CUDA — see paddlepaddle.org.cn
```

PaddlePaddle is installed separately on purpose: the wheel is CUDA-version-specific
and getting it wrong is the most common way to lose an afternoon.

Background on the data and the architecture choice: `docs/dataset-audit.html`,
`docs/run-a-research.html`.


## 0 · Environment


In [ ]:
import os, subprocess, sys, json, pathlib

REPO = pathlib.Path.cwd()
while not (REPO / 'ml' / 'prepare_dataset.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / 'ml').exists(), 'run this notebook from inside the repo'
os.chdir(REPO)
print('repo:', REPO)

def sh(cmd, **kw):
    """Run a shell command, streaming output live."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, **kw)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv')

In [ ]:
import paddle
print('paddle', paddle.__version__)
paddle.utils.run_check()

## 1 · Data

~331 MB from Roboflow. Both cells are idempotent — safe to re-run.


In [ ]:
assert sh('bash ml/download_dataset.sh') == 0

In [ ]:
assert sh('python ml/prepare_dataset.py '
          '--coco data/raw/train/_annotations.coco.json '
          '--images data/raw/train '
          '--out data/rec') == 0

### 1a · Verify the split

The notebook this replaces split on COCO `image_id`. Roboflow emits ~3 augmented
copies of each receipt, each with its own `image_id`, so copies of the same nota
landed in train, val *and* test — **96.1% of validation crops were contaminated**.

This asserts that can't happen, independently of the preparation script's own check.


In [ ]:
import itertools, re, collections

man = json.loads(pathlib.Path('data/rec/split_manifest.json').read_text())
r = man['receipts']
for a, b in itertools.combinations(('train','val','test'), 2):
    shared = set(r[a]) & set(r[b])
    assert not shared, f'LEAK: {len(shared)} receipts in both {a} and {b}'

# independent check: trace every crop back to its source receipt
coco = json.loads(pathlib.Path('data/raw/train/_annotations.coco.json').read_text())
src = {i['id']: re.match(r'^(.*?)_jpg\.rf\.', i['file_name']).group(1) for i in coco['images']}
for split in ('train','val','test'):
    ids = {int(l.split('/')[1].split('_')[0]) for l in open(f'data/rec/{split}_rec.txt')}
    stray = {src[i] for i in ids} - set(r[split])
    assert not stray, f'{split}: {len(stray)} crops from outside its split'

print('split verified disjoint —', man['grouped_by'])
for s in ('train','val','test'):
    n = sum(1 for _ in open(f'data/rec/{s}_rec.txt'))
    print(f'  {s:<6} {len(r[s]):>4} receipts  {n:>6} crops')
if man['grouped_by'] != 'writer':
    print('\nNOTE: grouped by receipt, not writer. The same 2-3 hands appear in every')
    print('      split, so val accuracy overstates real performance. See README.')

### 1b · Eyeball the crops

Cheapest possible check that cropping and the label mapping actually line up.


In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

rows = [l.rstrip('\n').split('\t') for l in open('data/rec/train_rec.txt')]
sample = random.Random(0).sample(rows, 24)

fig, axes = plt.subplots(4, 6, figsize=(15, 6))
for ax, (rel, text) in zip(axes.ravel(), sample):
    ax.imshow(Image.open(f'data/rec/{rel}'))
    ax.set_title(text, fontsize=11)
    ax.axis('off')
plt.tight_layout(); plt.show()

lengths = collections.Counter(len(t) for _, t in rows)
print('label lengths:', dict(sorted(lengths.items())))
print('numeric labels: %.0f%%' % (100*sum(1 for _,t in rows if t.isdigit())/len(rows)))

## 2 · Fine-tune

Clones PaddleOCR at a pinned tag, fetches pretrained weights, refuses to start if
the split leaks, trains, then exports an inference model.

**Training takes hours.** If your Jupyter session might drop, run it detached from a
terminal instead and use the tail cell below to watch:

```bash
nohup bash ml/finetune_rec.sh ~/work > ~/work/train.log 2>&1 &
```


In [ ]:
WORK = pathlib.Path.home() / 'work'
WORK.mkdir(exist_ok=True)
rc = sh(f'bash ml/finetune_rec.sh {WORK}')
print('\nexit', rc)

### 2a · Watch a detached run (skip if you trained above)


In [ ]:
# sh(f'tail -n 40 -f {WORK}/train.log')

### 2b · Training curve


In [ ]:
import re
log = WORK / 'PaddleOCR' / 'output' / 'nota_rec_v5_mobile'
lines = []
for p in sorted(log.glob('train.log*')) + sorted(WORK.glob('train.log')):
    lines += p.read_text(errors='ignore').splitlines()

loss = [(int(m.group(1)), float(m.group(2)))
        for m in (re.search(r'iter: (\d+).*?loss: ([\d.]+)', l) for l in lines) if m]
acc  = [(int(m.group(1)), float(m.group(2)))
        for m in (re.search(r'iter: (\d+).*?acc: ([\d.]+)', l) for l in lines) if m]

if loss:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(*zip(*loss), lw=.8); ax[0].set_title('train loss'); ax[0].set_xlabel('iter')
    if acc:
        ax[1].plot(*zip(*acc), lw=.8, color='seagreen'); ax[1].set_title('eval acc')
    plt.tight_layout(); plt.show()
else:
    print('no log lines parsed yet — check', log)

## 3 · Evaluate on the held-out test split

Never seen during training or model selection.


In [ ]:
PO = WORK / 'PaddleOCR'
sh(f'cd {PO} && python tools/eval.py -c configs/rec/nota_rec_v5_mobile.yml '
   f'-o Global.checkpoints=./output/nota_rec_v5_mobile/best_accuracy '
   f'Eval.dataset.label_file_list=[./data/rec/test_rec.txt]')

### 3a · Digit-stratified error

PaddleOCR reports one overall accuracy. That hides the thing that matters: a wrong
letter in a product name is cosmetic, a wrong digit in a price silently corrupts
inventory. This splits character error rate by digits vs letters.


In [ ]:
from paddleocr import TextRecognition

INFER = PO / 'output' / 'nota_rec_v5_mobile_infer'
rec = TextRecognition(model_dir=str(INFER))

def cer(pred, gold):
    """Levenshtein distance normalised by reference length."""
    prev = list(range(len(gold) + 1))
    for i, p in enumerate(pred, 1):
        cur = [i]
        for j, g in enumerate(gold, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (p != g)))
        prev = cur
    return prev[-1] / max(len(gold), 1)

test = [l.rstrip('\n').split('\t') for l in open('data/rec/test_rec.txt')]
preds = []
for i in range(0, len(test), 256):                      # batched; predict() returns a list
    chunk = [f'data/rec/{rel}' for rel, _ in test[i:i+256]]
    preds += [r.get('rec_text') or '' for r in rec.predict(chunk)]
assert len(preds) == len(test), (len(preds), len(test))

buckets, exact, worst = {'digits': [], 'letters': [], 'all': []}, 0, []
for (rel, gold), pred in zip(test, preds):
    e = cer(pred, gold)
    buckets['all'].append(e)
    buckets['digits' if gold.isdigit() else 'letters'].append(e)
    exact += (pred == gold)
    if pred != gold:
        worst.append((e, gold, pred))

for k, v in buckets.items():
    if v:
        print(f'{k:<8} CER {sum(v)/len(v):6.2%}   n={len(v)}')
print(f'\nexact-match  {exact/len(test):.2%}  ({exact}/{len(test)} crops)')

print('\nworst misreads (gold -> predicted):')
for e, gold, pred in sorted(worst, reverse=True)[:15]:
    flag = '  <-- DIGIT' if gold.isdigit() else ''
    print(f'  {e:5.2f}  {gold!r:>12} -> {pred!r}{flag}')

## 4 · What next

The exported model is at `~/work/PaddleOCR/output/nota_rec_v5_mobile_infer/` —
that's what the OCR service will load.

1. **Writer-held-out split.** Label handwriting into a `source_id,writer_id` CSV and
   re-run `prepare_dataset.py --writer-map writers.csv`. Until then the number above
   is measured on the same few hands the model trained on.
2. **Ablation ladder** (`docs/run-a-research.html`): this is condition 1. Conditions 2
   and 3 — synthetic vocabulary vs synthetic handwriting style — are the experiment
   that decides the data strategy.
3. **INT8 quantization** for CPU serving, and measure what accuracy it costs.
